In [10]:
import nltk 
from nltk.tokenize import word_tokenize
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import gensim.downloader as api
import random

In [11]:
df = pd.read_csv("hf://datasets/gsri-18/ISEAR-dataset-complete/ISEAR_dataset_complete.csv")
df = df.drop(columns=["Unnamed: 2"])
content = df["content"]
print(content.head())
emotions = df["emotion"].unique()

# Representation 1: Tokenization
tokens = []
for element in df["content"]:
    tokenized_sentence = word_tokenize(element)
    tokens.append(tokenized_sentence)
df["tokens"] = tokens


0    On days when I feel close to my partner and ot...
1    Every time I imagine that someone I love or I ...
2    When I had been obviously unjustly treated and...
3    When I think about the short time that we live...
4    At a gathering I found myself involuntarily si...
Name: content, dtype: str


In [12]:
# Embedding
from gensim.models import Word2Vec

model_CBOW = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=0,
    epochs=10
)

model_S = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10
)

In [13]:
# All unique words in your dataset
dataset_vocab = set(
    word
    for sentence in df["tokens"]
    for word in sentence
)

# Words represented in the Word2Vec model
model_vocab = set(model_CBOW.wv.key_to_index.keys())

# Words present in both vocabularies
covered_words = dataset_vocab.intersection(model_vocab)

# Words missing from the model
oov_words = dataset_vocab - model_vocab

# Calculate coverage
coverage = len(covered_words) / len(dataset_vocab) * 100

print(f"Dataset vocabulary: {len(dataset_vocab)}")
print(f"Model vocabulary: {len(model_vocab)}")
print(f"Covered words: {len(covered_words)}")
print(f"OOV words: {len(oov_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")

print("Missing words:", list(oov_words)[:30])

Dataset vocabulary: 10173
Model vocabulary: 4971
Covered words: 4971
OOV words: 5202
Vocabulary coverage: 48.86%
Missing words: ['scoring', 'saleswoman', 'Speech', 'humanoids', 'cuddled', 'socialist', 'hitch-hiked', 'tipped', 'crabs', 'shirked', 'Ullevi', 'tip', 'definite', 'operations', 'sagged', 'nodule', 'violating', 'adked', 'hopped', 'piles', 'lanes', 'discontinue', 'progressing', 'plead', 'slimily', 'unlimited', 'tribal', 'Catholicism', 'mourning', 'fatigue']


In [14]:
# All unique words in your dataset
dataset_vocab = set(
    word
    for sentence in df["tokens"]
    for word in sentence
)

# Words represented in the Word2Vec model
model_vocab = set(model_S.wv.key_to_index.keys())

# Words present in both vocabularies
covered_words = dataset_vocab.intersection(model_vocab)

# Words missing from the model
oov_words = dataset_vocab - model_vocab

# Calculate coverage
coverage = len(covered_words) / len(dataset_vocab) * 100

print(f"Dataset vocabulary: {len(dataset_vocab)}")
print(f"Model vocabulary: {len(model_vocab)}")
print(f"Covered words: {len(covered_words)}")
print(f"OOV words: {len(oov_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")

print("Missing words:", list(oov_words)[:30])

Dataset vocabulary: 10173
Model vocabulary: 4971
Covered words: 4971
OOV words: 5202
Vocabulary coverage: 48.86%
Missing words: ['scoring', 'saleswoman', 'Speech', 'humanoids', 'cuddled', 'socialist', 'hitch-hiked', 'tipped', 'crabs', 'shirked', 'Ullevi', 'tip', 'definite', 'operations', 'sagged', 'nodule', 'violating', 'adked', 'hopped', 'piles', 'lanes', 'discontinue', 'progressing', 'plead', 'slimily', 'unlimited', 'tribal', 'Catholicism', 'mourning', 'fatigue']


In [15]:
total_tokens = sum(
    len(sentence)
    for sentence in df["tokens"]
)

covered_tokens = sum(
    1
    for sentence in df["tokens"]
    for word in sentence
    if word in model_CBOW.wv
)

token_coverage = covered_tokens / total_tokens * 100

print(f"Token-level coverage CBOW: {token_coverage:.2f}%")

Token-level coverage CBOW: 97.18%


In [16]:
total_tokens = sum(
    len(sentence)
    for sentence in df["tokens"]
)

covered_tokens = sum(
    1
    for sentence in df["tokens"]
    for word in sentence
    if word in model_S.wv
)

token_coverage = covered_tokens / total_tokens * 100

print(f"Token-level coverage S: {token_coverage:.2f}%")

Token-level coverage S: 97.18%


In [33]:
print(model_CBOW.wv.most_similar("summer", topn=10))
print(model_S.wv.most_similar("summer", topn=10))

[('tennis', 0.9770531058311462), ('Getting', 0.969646692276001), ('Bombay', 0.962776780128479), ('army', 0.9608882665634155), ('Summer', 0.9592043161392212), ('basket', 0.9587479829788208), ('traffic', 0.9580353498458862), ('May', 0.957384467124939), ('vomitted', 0.9572111368179321), ('passing', 0.9570576548576355)]
[('Summer', 0.8829002976417542), ('carnaval', 0.8813093900680542), ('Last', 0.865871787071228), ('Sweden', 0.8643437027931213), ('journey', 0.8610979914665222), ('army', 0.8598523139953613), ('tennis', 0.8561864495277405), ('French', 0.8558905124664307), ('autumn', 0.8527909517288208), ('vacations', 0.852191150188446)]


In [18]:
print([word for word in model_vocab if len(word) <= 2])
print([word for word in dataset_vocab if len(word) <= 2])

['TA', 'PY', 'aa', 'if', 'up', 'in', 'be', 'We', ']', 'on', '%', 'e', 'Ca', '3', 'B', '.', 'TV', 'On', 'By', '5', '80', 'LP', 'we', '$', 'He', 'go', 'I', 'In', '50', 'by', 'No', '(', 'it', 'My', '/', '``', '20', 'OK', '6', '?', '17', 'so', '14', '21', '40', '[', ',', 'wo', 'km', 'do', '15', '84', 'Km', 'i', '7', '22', 'or', ';', '60', ')', '35', 'is', 'am', 'X', 'TE', 'As', 'NO', 'Ye', "'m", 'C', 'm', 'Is', 'ok', 'A', '4', "''", 'At', 'of', '12', 'An', '30', 'a', 'F4', 'ca', 'us', 'no', '11', 'me', '9', 'Oh', 'an', 'to', '18', '!', "'s", 'II', '16', '19', '&', 'as', 'C+', '13', '8', "'d", "'", 'pm', 'So', '2A', 'D', '10', 'I.', 'my', 'B+', 'Mr', 'To', 'he', 'Of', '1', '2', 'at', ':', 'It', 'If', 'ex', '-']
['TA', 'PY', 'aa', 'if', '53', 'up', 'in', 'be', 'We', ']', 'on', '%', 'e', 'Ca', 'o', 'P3', 'E', 'T', '3', 'B', '.', 'n', 'TV', 'On', 'By', 'cm', '5', '23', 'WD', '2C', '80', 'A+', 'ne', 'LP', '1A', 'we', '$', 'He', 'go', 'I', 'In', '50', 'by', 'No', '(', 'it', 'CS', 'My', '/', '``'

In [20]:
# Pooling using mean
def mean_pool(tokens, word_vectors):
    vectors = [
        word_vectors[token]
        for token in tokens
        if token in word_vectors
    ]

    if not vectors:
        return np.zeros(word_vectors.vector_size)

    return np.mean(vectors, axis=0)

# Pooling using max
def max_pool(tokens, word_vectors):
    vectors = [
        word_vectors[token]
        for token in tokens
        if token in word_vectors
    ]

    if not vectors:
        return np.zeros(word_vectors.vector_size)

    return np.max(vectors, axis=0)

# Create a mean and max pooled vector for each document, for both embedding models
X_mean_cbow = np.vstack([
    mean_pool(tokens, model_CBOW.wv)
    for tokens in df["tokens"]
])

X_mean_skipgram = np.vstack([
    mean_pool(tokens, model_S.wv)
    for tokens in df["tokens"]
])

X_max_cbow = np.vstack([
    max_pool(tokens, model_CBOW.wv)
    for tokens in df["tokens"]
])

X_max_skipgram = np.vstack([
    max_pool(tokens, model_S.wv)
    for tokens in df["tokens"]
])

# Vector shapes should be the same
print(X_mean_cbow.shape)
print(X_mean_skipgram.shape)
print(X_max_cbow.shape)
print(X_max_skipgram.shape)


(7516, 100)
(7516, 100)
(7516, 100)
(7516, 100)


In [27]:
# Training classifiers
y = df["emotion"]

implementations = {
    "CBOW + mean": X_mean_cbow,
    "Skip-gram + mean": X_mean_skipgram,
    "CBOW + max": X_max_cbow,
    "Skip-gram + max": X_max_skipgram,
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = {}

for name, X in implementations.items():
    classifier = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    scores = cross_val_score(
        classifier,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    results[name] = scores
    print(
        f"{name}: "
        f"mean={scores.mean():.4f}, "
        f"std={scores.std():.4f}"
    )

CBOW + mean: mean=0.3527, std=0.0133
Skip-gram + mean: mean=0.4271, std=0.0083
CBOW + max: mean=0.3268, std=0.0064
Skip-gram + max: mean=0.3703, std=0.0056


In [28]:
# Now trying the same but using Glove instead
glove = api.load("glove-wiki-gigaword-100")
vector_size = glove.vector_size

# Glove works in lowercase
df["glove_tokens"] = [
    [token.lower() for token in tokens]
    for tokens in df["tokens"]
]

# Pooling functions but with glove:
def glove_mean_pool(tokens, embeddings):
    vectors = [
        embeddings[token]
        for token in tokens
        if token in embeddings
    ]

    if not vectors:
        return np.zeros(embeddings.vector_size)

    return np.mean(vectors, axis=0)

def glove_max_pool(tokens, embeddings):
    vectors = [
        embeddings[token]
        for token in tokens
        if token in embeddings
    ]

    if not vectors:
        return np.zeros(embeddings.vector_size)

    return np.max(vectors, axis=0)

# Create a mean and max pooled vector for each document, for both embedding models
X_glove_mean = np.vstack([
    glove_mean_pool(tokens, glove)
    for tokens in df["glove_tokens"]
])

X_glove_max = np.vstack([
    glove_max_pool(tokens, glove)
    for tokens in df["glove_tokens"]
])

print(X_glove_mean.shape)
print(X_glove_max.shape)

(7516, 100)
(7516, 100)


In [29]:
# Glove coverage
glove_covered_tokens = sum(
    token in glove
    for tokens in df["glove_tokens"]
    for token in tokens
)

glove_total_tokens = sum(
    len(tokens)
    for tokens in df["glove_tokens"]
)

print(
    f"GloVe token coverage: "
    f"{glove_covered_tokens / glove_total_tokens:.2%}"
)

GloVe token coverage: 99.37%


In [30]:
# Unique words in the dataset
dataset_vocab = {
    word
    for sentence in df["glove_tokens"]
    for word in sentence
}

# Words available in the GloVe model
model_vocab = set(glove.key_to_index)

covered_words = dataset_vocab & model_vocab
oov_words = dataset_vocab - model_vocab

coverage = (
    len(covered_words) / len(dataset_vocab) * 100
    if dataset_vocab else 0
)

print(f"Dataset vocabulary: {len(dataset_vocab)}")
print(f"GloVe vocabulary: {len(model_vocab)}")
print(f"Covered words: {len(covered_words)}")
print(f"OOV words: {len(oov_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")
print("Missing words:", sorted(oov_words)[:30])

Dataset vocabulary: 9388
GloVe vocabulary: 400000
Covered words: 8449
OOV words: 939
Vocabulary coverage: 90.00%
Missing words: ['-down', '-rooms', '-that', '......', '........', '.........', '..........', '.the', '10t', '10yrs', '11-months', '110km/h', '12/83', '1bs', '21/30', '3000/', '4-letter', '4th/5th', '9.30/10', 'a+', 'above-board', 'abruptely', 'absailing', 'abscence', 'absent-mindedness', 'acccouting', 'accentance', 'acceptted', 'accidentaly', 'accquaintances']


In [34]:
print(glove.most_similar("summer", topn=10))


[('winter', 0.8896950483322144), ('spring', 0.8580390214920044), ('autumn', 0.7742397785186768), ('weekend', 0.7385303378105164), ('year', 0.7348463535308838), ('days', 0.7250120043754578), ('beginning', 0.7218300104141235), ('during', 0.7205086946487427), ('season', 0.7031364440917969), ('day', 0.701505720615387)]


In [14]:
# Training classifiers
y = df["emotion"]

implementations = {
    "GloVe mean": X_glove_mean,
    "GloVe max": X_glove_max,
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for name, X in implementations.items():
    classifier = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    scores = cross_val_score(
        classifier,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    print(
        f"{name}: "
        f"mean={scores.mean():.4f}, "
        f"std={scores.std():.4f}"
    )

GloVe mean: mean=0.4864, std=0.0164
GloVe max: mean=0.4211, std=0.0099


In [24]:
import sys
import os
results_folder = "../Data/Results"
os.makedirs(results_folder, exist_ok=True)

In [25]:
# Willen jullie dit testen a.u.b.?
sys.path.insert(0, os.path.abspath(".."))
import jax
import numpyro
import part2
import itertools

glove = api.load("glove-wiki-gigaword-100", return_path=True)
trained_emb_model = part2.clslib.PretrainedEmbeddings(glove, 100, skip_first_line=True)
tokenizer_for_lstm = part2.clslib.BasicTokenizer(
    True, False, False, 'english', trained_emb_model)

def cross_validation_split(pairs, k, seed=0):
    """
    Shuffles a list of labelled texts and splits the result into k folds.
    
    pairs: lists of tuples (text, label)
    k: number of folds
    seed: for reproducible shuffling
    """
    
    # Shuffle the pairs
    random.seed(seed)
    pairs = list(pairs)
    ids = list(range(len(pairs)))
    random.shuffle(ids)

    # Split without coercing (text, int) tuples into a string array
    ids_parts = np.array_split(np.array(ids, dtype=object), k)
    
    # Create a dictionary to store the training and validation sets for each fold
    cross_validation_sets = {}
    for i in range(k):
        validation_set = [pairs[j] for j in ids_parts[i]] 
        training_ids = list(itertools.chain.from_iterable(part for k, part in enumerate(ids_parts) if k != i))
        training_set = [pairs[j] for j in training_ids]
        cross_validation_sets[i] = {
            "training": training_set, 
            "validation": validation_set, 
            'training_ids': training_ids,   # these are not often needed, but we return it just in case
            'validation_ids': ids_parts[i]  # these are not often needed, but we return it just in case
        }

    return cross_validation_sets

pairs=list(zip(df["content"], df["emotion"].map(list(emotions).index)))
splits = cross_validation_split(pairs, k=5)

part2.experiment.cross_validation(
        outcome_space=emotions,
        data_splits=splits,
        tokenizer=tokenizer_for_lstm,
        emb_model=trained_emb_model,
        enc_type='lstm', 
        enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.},
        regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
        opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 20},    
        device='cpu', #'cpu'
        save_to=f"{results_folder}/lstm",
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.184 seconds
Dataloader done! Time taken: 0.022 seconds


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9367 | train acc: 0.1705 | val loss: 1.9062 | val acc: 0.1948
Epoch 02 | train loss: 1.8020 | train acc: 0.2650 | val loss: 1.6988 | val acc: 0.3191
Epoch 03 | train loss: 1.6729 | train acc: 0.3397 | val loss: 1.6293 | val acc: 0.3657
Epoch 04 | train loss: 1.5765 | train acc: 0.3897 | val loss: 1.5326 | val acc: 0.4189
Epoch 05 | train loss: 1.5012 | train acc: 0.4213 | val loss: 1.4793 | val acc: 0.4608
Epoch 06 | train loss: 1.4546 | train acc: 0.4481 | val loss: 1.4411 | val acc: 0.4608
Epoch 07 | train loss: 1.3999 | train acc: 0.4697 | val loss: 1.4790 | val acc: 0.4535
Epoch 08 | train loss: 1.3703 | train acc: 0.4917 | val loss: 1.3759 | val acc: 0.4960
Epoch 09 | train loss: 1.3165 | train acc: 0.5141 | val loss: 1.3554 | val acc: 0.5033
Epoch 10 | train loss: 1.2806 | train acc: 0.5328 | val loss: 1.3206 | val acc: 0.5193
Epoch 11 | train loss: 1.2467 | train acc: 0.5492 | val loss: 1.3244 | val acc: 0.5246
Epoch 12 | train loss: 1.2101 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9297 | train acc: 0.1731 | val loss: 1.8513 | val acc: 0.2428
Epoch 02 | train loss: 1.7807 | train acc: 0.2884 | val loss: 1.7165 | val acc: 0.2954
Epoch 03 | train loss: 1.6629 | train acc: 0.3389 | val loss: 1.5936 | val acc: 0.3866
Epoch 04 | train loss: 1.5788 | train acc: 0.3900 | val loss: 1.5755 | val acc: 0.3859
Epoch 05 | train loss: 1.5293 | train acc: 0.4098 | val loss: 1.5027 | val acc: 0.4218
Epoch 06 | train loss: 1.4810 | train acc: 0.4326 | val loss: 1.4877 | val acc: 0.4331
Epoch 07 | train loss: 1.4318 | train acc: 0.4527 | val loss: 1.4629 | val acc: 0.4458
Epoch 08 | train loss: 1.3931 | train acc: 0.4745 | val loss: 1.4438 | val acc: 0.4604
Epoch 09 | train loss: 1.3610 | train acc: 0.4893 | val loss: 1.3895 | val acc: 0.4857
Epoch 10 | train loss: 1.3083 | train acc: 0.5081 | val loss: 1.3667 | val acc: 0.5096
Epoch 11 | train loss: 1.2800 | train acc: 0.5209 | val loss: 1.3359 | val acc: 0.5183
Epoch 12 | train loss: 1.2577 | train acc: 

... (3 outputs left out)

Epoch 01 | train loss: 1.9215 | train acc: 0.1758 | val loss: 1.8569 | val acc: 0.2369
Epoch 02 | train loss: 1.7985 | train acc: 0.2688 | val loss: 1.7436 | val acc: 0.2881
Epoch 03 | train loss: 1.6841 | train acc: 0.3256 | val loss: 1.6560 | val acc: 0.3407
Epoch 04 | train loss: 1.5995 | train acc: 0.3700 | val loss: 1.5711 | val acc: 0.4059
Epoch 05 | train loss: 1.5155 | train acc: 0.4196 | val loss: 1.5130 | val acc: 0.4365
Epoch 06 | train loss: 1.4634 | train acc: 0.4474 | val loss: 1.4594 | val acc: 0.4511
Epoch 07 | train loss: 1.4065 | train acc: 0.4733 | val loss: 1.4414 | val acc: 0.4770
Epoch 08 | train loss: 1.3709 | train acc: 0.4948 | val loss: 1.4209 | val acc: 0.4830
Epoch 09 | train loss: 1.3358 | train acc: 0.5072 | val loss: 1.3640 | val acc: 0.5143
Epoch 10 | train loss: 1.3086 | train acc: 0.5134 | val loss: 1.3589 | val acc: 0.5083
Epoch 11 | train loss: 1.2672 | train acc: 0.5375 | val loss: 1.3244 | val acc: 0.5243
Epoch 12 | train loss: 1.2360 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9330 | train acc: 0.1833 | val loss: 1.8759 | val acc: 0.2249
Epoch 02 | train loss: 1.8013 | train acc: 0.2880 | val loss: 1.7064 | val acc: 0.3320
Epoch 03 | train loss: 1.6592 | train acc: 0.3542 | val loss: 1.6007 | val acc: 0.3912
Epoch 04 | train loss: 1.5631 | train acc: 0.4026 | val loss: 1.5695 | val acc: 0.3972
Epoch 05 | train loss: 1.4932 | train acc: 0.4299 | val loss: 1.5062 | val acc: 0.4325
Epoch 06 | train loss: 1.4466 | train acc: 0.4514 | val loss: 1.4721 | val acc: 0.4464
Epoch 07 | train loss: 1.3902 | train acc: 0.4701 | val loss: 1.4378 | val acc: 0.4677
Epoch 08 | train loss: 1.3637 | train acc: 0.4876 | val loss: 1.4125 | val acc: 0.4770
Epoch 09 | train loss: 1.3330 | train acc: 0.5017 | val loss: 1.3943 | val acc: 0.4797
Epoch 10 | train loss: 1.3072 | train acc: 0.5132 | val loss: 1.3813 | val acc: 0.4817
Epoch 11 | train loss: 1.2812 | train acc: 0.5264 | val loss: 1.3642 | val acc: 0.4963
Epoch 12 | train loss: 1.2505 | train acc: 

{'models': OrderedDict([(0, '../Data/Results/lstm/fold-0/model'),
              (1, '../Data/Results/lstm/fold-1/model'),
              (2, '../Data/Results/lstm/fold-2/model'),
              (3, '../Data/Results/lstm/fold-3/model'),
              (4, '../Data/Results/lstm/fold-4/model')]),
 'predictions': OrderedDict([(0,
               array([4, 1, 6, ..., 3, 3, 0], shape=(1504,))),
              (1, array([5, 1, 1, ..., 3, 4, 0], shape=(1503,))),
              (2, array([5, 0, 0, ..., 2, 0, 2], shape=(1503,))),
              (3, array([0, 4, 6, ..., 0, 1, 4], shape=(1503,))),
              (4, array([6, 4, 6, ..., 0, 0, 4], shape=(1503,)))]),
 'accuracies': OrderedDict([(0, 0.5691489361702128),
              (1, 0.5508982035928144),
              (2, 0.5462408516300732),
              (3, 0.5515635395874917),
              (4, 0.5455755156353959)])}

In [41]:
# Batch size 64 -> 32
part2.experiment.cross_validation(
        outcome_space=emotions,
        data_splits=splits,
        tokenizer=tokenizer_for_lstm,
        emb_model=trained_emb_model,
        enc_type='lstm', 
        enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.},
        regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
        opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 32, 'num_epochs': 20},    
        device='cpu', #'cpu'
        save_to=f"{results_folder}/lstm",
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.285 seconds
Dataloader done! Time taken: 0.029 seconds


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9063 | train acc: 0.1905 | val loss: 1.7830 | val acc: 0.2932
Epoch 02 | train loss: 1.7301 | train acc: 0.3170 | val loss: 1.6045 | val acc: 0.3743
Epoch 03 | train loss: 1.5573 | train acc: 0.4010 | val loss: 1.4822 | val acc: 0.4375
Epoch 04 | train loss: 1.4609 | train acc: 0.4436 | val loss: 1.4260 | val acc: 0.4628
Epoch 05 | train loss: 1.3933 | train acc: 0.4682 | val loss: 1.3856 | val acc: 0.4767
Epoch 06 | train loss: 1.3614 | train acc: 0.4844 | val loss: 1.3495 | val acc: 0.5100
Epoch 07 | train loss: 1.3063 | train acc: 0.5153 | val loss: 1.3247 | val acc: 0.5113
Epoch 08 | train loss: 1.2628 | train acc: 0.5314 | val loss: 1.3110 | val acc: 0.5279
Epoch 09 | train loss: 1.2277 | train acc: 0.5567 | val loss: 1.3316 | val acc: 0.5180
Epoch 10 | train loss: 1.1991 | train acc: 0.5709 | val loss: 1.2723 | val acc: 0.5366
Epoch 11 | train loss: 1.1700 | train acc: 0.5722 | val loss: 1.2831 | val acc: 0.5419
Epoch 12 | train loss: 1.1238 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9250 | train acc: 0.1856 | val loss: 1.8572 | val acc: 0.2422
Epoch 02 | train loss: 1.7161 | train acc: 0.3288 | val loss: 1.6344 | val acc: 0.3812
Epoch 03 | train loss: 1.5689 | train acc: 0.3920 | val loss: 1.5293 | val acc: 0.4245
Epoch 04 | train loss: 1.4835 | train acc: 0.4241 | val loss: 1.4579 | val acc: 0.4345
Epoch 05 | train loss: 1.4096 | train acc: 0.4555 | val loss: 1.4295 | val acc: 0.4531
Epoch 06 | train loss: 1.3609 | train acc: 0.4828 | val loss: 1.4257 | val acc: 0.4551
Epoch 07 | train loss: 1.3113 | train acc: 0.5014 | val loss: 1.3560 | val acc: 0.4970
Epoch 08 | train loss: 1.2753 | train acc: 0.5209 | val loss: 1.3474 | val acc: 0.4963
Epoch 09 | train loss: 1.2434 | train acc: 0.5415 | val loss: 1.3272 | val acc: 0.5050
Epoch 10 | train loss: 1.1982 | train acc: 0.5608 | val loss: 1.3278 | val acc: 0.5096
Epoch 11 | train loss: 1.1691 | train acc: 0.5696 | val loss: 1.3237 | val acc: 0.5116
Epoch 12 | train loss: 1.1511 | train acc: 

... (3 outputs left out)

Epoch 01 | train loss: 1.8951 | train acc: 0.2059 | val loss: 1.7802 | val acc: 0.2961
Epoch 02 | train loss: 1.6994 | train acc: 0.3373 | val loss: 1.6418 | val acc: 0.3500
Epoch 03 | train loss: 1.5796 | train acc: 0.3897 | val loss: 1.5933 | val acc: 0.3952
Epoch 04 | train loss: 1.4883 | train acc: 0.4342 | val loss: 1.4985 | val acc: 0.4411
Epoch 05 | train loss: 1.4001 | train acc: 0.4723 | val loss: 1.4487 | val acc: 0.4704
Epoch 06 | train loss: 1.3399 | train acc: 0.5054 | val loss: 1.3629 | val acc: 0.5136
Epoch 07 | train loss: 1.2897 | train acc: 0.5305 | val loss: 1.3277 | val acc: 0.5150
Epoch 08 | train loss: 1.2585 | train acc: 0.5383 | val loss: 1.2973 | val acc: 0.5383
Epoch 09 | train loss: 1.2073 | train acc: 0.5626 | val loss: 1.3004 | val acc: 0.5409
Epoch 10 | train loss: 1.1906 | train acc: 0.5729 | val loss: 1.2992 | val acc: 0.5442
Epoch 11 | train loss: 1.1459 | train acc: 0.5872 | val loss: 1.2525 | val acc: 0.5675
Epoch 12 | train loss: 1.1294 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9094 | train acc: 0.1992 | val loss: 1.7765 | val acc: 0.2562
Epoch 02 | train loss: 1.7158 | train acc: 0.3188 | val loss: 1.6247 | val acc: 0.3772
Epoch 03 | train loss: 1.5733 | train acc: 0.3981 | val loss: 1.5702 | val acc: 0.3852
Epoch 04 | train loss: 1.4722 | train acc: 0.4387 | val loss: 1.4689 | val acc: 0.4291
Epoch 05 | train loss: 1.3926 | train acc: 0.4740 | val loss: 1.4606 | val acc: 0.4484
Epoch 06 | train loss: 1.3310 | train acc: 0.5051 | val loss: 1.3793 | val acc: 0.4850
Epoch 07 | train loss: 1.2815 | train acc: 0.5358 | val loss: 1.3740 | val acc: 0.4904
Epoch 08 | train loss: 1.2457 | train acc: 0.5480 | val loss: 1.3563 | val acc: 0.4923
Epoch 09 | train loss: 1.1961 | train acc: 0.5668 | val loss: 1.3127 | val acc: 0.5316
Epoch 10 | train loss: 1.1607 | train acc: 0.5859 | val loss: 1.2843 | val acc: 0.5303
Epoch 11 | train loss: 1.1441 | train acc: 0.5881 | val loss: 1.2781 | val acc: 0.5403
Epoch 12 | train loss: 1.1125 | train acc: 

{'models': OrderedDict([(0, '../Data/Results/lstm/fold-0/model'),
              (1, '../Data/Results/lstm/fold-1/model'),
              (2, '../Data/Results/lstm/fold-2/model'),
              (3, '../Data/Results/lstm/fold-3/model'),
              (4, '../Data/Results/lstm/fold-4/model')]),
 'predictions': OrderedDict([(0,
               array([4, 1, 6, ..., 6, 3, 3], shape=(1504,))),
              (1, array([5, 1, 1, ..., 3, 4, 0], shape=(1503,))),
              (2, array([2, 0, 0, ..., 5, 0, 2], shape=(1503,))),
              (3, array([0, 4, 6, ..., 3, 1, 4], shape=(1503,))),
              (4, array([6, 2, 6, ..., 0, 3, 4], shape=(1503,)))]),
 'accuracies': OrderedDict([(0, 0.569813829787234),
              (1, 0.5568862275449101),
              (2, 0.552228875582169),
              (3, 0.573519627411843),
              (4, 0.5701929474384564)])}

In [42]:
# learning rate 1e-3 -> 1e-4
part2.experiment.cross_validation(
        outcome_space=emotions,
        data_splits=splits,
        tokenizer=tokenizer_for_lstm,
        emb_model=trained_emb_model,
        enc_type='lstm', 
        enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.},
        regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
        opt_kwargs={'learning_rate': 1e-4, 'weight_decay': 1e-4, 'batch_size': 32, 'num_epochs': 20},    
        device='cpu', #'cpu'
        save_to=f"{results_folder}/lstm",
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.266 seconds
Dataloader done! Time taken: 0.025 seconds


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9484 | train acc: 0.1459 | val loss: 1.9447 | val acc: 0.1642
Epoch 02 | train loss: 1.9453 | train acc: 0.1396 | val loss: 1.9428 | val acc: 0.1815
Epoch 03 | train loss: 1.9428 | train acc: 0.1592 | val loss: 1.9398 | val acc: 0.2021
Epoch 04 | train loss: 1.9384 | train acc: 0.1791 | val loss: 1.9318 | val acc: 0.2400
Epoch 05 | train loss: 1.9248 | train acc: 0.1966 | val loss: 1.9099 | val acc: 0.2380
Epoch 06 | train loss: 1.8928 | train acc: 0.2186 | val loss: 1.8602 | val acc: 0.2527
Epoch 07 | train loss: 1.8527 | train acc: 0.2422 | val loss: 1.8195 | val acc: 0.2753
Epoch 08 | train loss: 1.8127 | train acc: 0.2625 | val loss: 1.7884 | val acc: 0.2839
Epoch 09 | train loss: 1.7867 | train acc: 0.2746 | val loss: 1.7713 | val acc: 0.3125
Epoch 10 | train loss: 1.7651 | train acc: 0.2876 | val loss: 1.7432 | val acc: 0.3059
Epoch 11 | train loss: 1.7480 | train acc: 0.2991 | val loss: 1.7269 | val acc: 0.3118
Epoch 12 | train loss: 1.7336 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9450 | train acc: 0.1587 | val loss: 1.9393 | val acc: 0.2083
Epoch 02 | train loss: 1.9345 | train acc: 0.1903 | val loss: 1.9226 | val acc: 0.2176
Epoch 03 | train loss: 1.8993 | train acc: 0.2210 | val loss: 1.8665 | val acc: 0.2369
Epoch 04 | train loss: 1.8538 | train acc: 0.2327 | val loss: 1.8288 | val acc: 0.2575
Epoch 05 | train loss: 1.8236 | train acc: 0.2511 | val loss: 1.8124 | val acc: 0.2675
Epoch 06 | train loss: 1.8003 | train acc: 0.2664 | val loss: 1.7908 | val acc: 0.3021
Epoch 07 | train loss: 1.7787 | train acc: 0.2734 | val loss: 1.7709 | val acc: 0.3167
Epoch 08 | train loss: 1.7592 | train acc: 0.2809 | val loss: 1.7503 | val acc: 0.3140
Epoch 09 | train loss: 1.7380 | train acc: 0.3002 | val loss: 1.7334 | val acc: 0.3200
Epoch 10 | train loss: 1.7121 | train acc: 0.3220 | val loss: 1.7176 | val acc: 0.3194
Epoch 11 | train loss: 1.6944 | train acc: 0.3245 | val loss: 1.7082 | val acc: 0.3227
Epoch 12 | train loss: 1.6790 | train acc: 

... (3 outputs left out)

Epoch 01 | train loss: 1.9489 | train acc: 0.1444 | val loss: 1.9474 | val acc: 0.1444
Epoch 02 | train loss: 1.9446 | train acc: 0.1548 | val loss: 1.9441 | val acc: 0.1484
Epoch 03 | train loss: 1.9399 | train acc: 0.1808 | val loss: 1.9364 | val acc: 0.1903
Epoch 04 | train loss: 1.9223 | train acc: 0.2127 | val loss: 1.9062 | val acc: 0.2236
Epoch 05 | train loss: 1.8668 | train acc: 0.2473 | val loss: 1.8411 | val acc: 0.2482
Epoch 06 | train loss: 1.8142 | train acc: 0.2681 | val loss: 1.8027 | val acc: 0.2854
Epoch 07 | train loss: 1.7812 | train acc: 0.2761 | val loss: 1.7814 | val acc: 0.2854
Epoch 08 | train loss: 1.7552 | train acc: 0.2922 | val loss: 1.7594 | val acc: 0.2967
Epoch 09 | train loss: 1.7357 | train acc: 0.3003 | val loss: 1.7450 | val acc: 0.3154
Epoch 10 | train loss: 1.7134 | train acc: 0.3110 | val loss: 1.7358 | val acc: 0.3280
Epoch 11 | train loss: 1.6980 | train acc: 0.3245 | val loss: 1.7193 | val acc: 0.3307
Epoch 12 | train loss: 1.6837 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9531 | train acc: 0.1397 | val loss: 1.9458 | val acc: 0.1544
Epoch 02 | train loss: 1.9473 | train acc: 0.1422 | val loss: 1.9438 | val acc: 0.1717
Epoch 03 | train loss: 1.9434 | train acc: 0.1545 | val loss: 1.9415 | val acc: 0.1910
Epoch 04 | train loss: 1.9381 | train acc: 0.1913 | val loss: 1.9343 | val acc: 0.2089
Epoch 05 | train loss: 1.9213 | train acc: 0.2085 | val loss: 1.9041 | val acc: 0.2282
Epoch 06 | train loss: 1.8779 | train acc: 0.2406 | val loss: 1.8541 | val acc: 0.2495
Epoch 07 | train loss: 1.8374 | train acc: 0.2623 | val loss: 1.8117 | val acc: 0.2735
Epoch 08 | train loss: 1.7946 | train acc: 0.2855 | val loss: 1.7652 | val acc: 0.3027
Epoch 09 | train loss: 1.7531 | train acc: 0.3023 | val loss: 1.7403 | val acc: 0.2994
Epoch 10 | train loss: 1.7213 | train acc: 0.3105 | val loss: 1.6923 | val acc: 0.3273
Epoch 11 | train loss: 1.6972 | train acc: 0.3238 | val loss: 1.6708 | val acc: 0.3420
Epoch 12 | train loss: 1.6665 | train acc: 

{'models': OrderedDict([(0, '../Data/Results/lstm/fold-0/model'),
              (1, '../Data/Results/lstm/fold-1/model'),
              (2, '../Data/Results/lstm/fold-2/model'),
              (3, '../Data/Results/lstm/fold-3/model'),
              (4, '../Data/Results/lstm/fold-4/model')]),
 'predictions': OrderedDict([(0,
               array([4, 1, 6, ..., 6, 2, 0], shape=(1504,))),
              (1, array([5, 5, 1, ..., 3, 4, 0], shape=(1503,))),
              (2, array([6, 0, 1, ..., 4, 0, 2], shape=(1503,))),
              (3, array([0, 1, 4, ..., 0, 2, 2], shape=(1503,))),
              (4, array([4, 6, 6, ..., 0, 3, 4], shape=(1503,)))]),
 'accuracies': OrderedDict([(0, 0.35970744680851063),
              (1, 0.3666001330671989),
              (2, 0.3473053892215569),
              (3, 0.3812375249500998),
              (4, 0.37258815701929476)])}

In [43]:
# hidden size 1&2 50 -> 100
part2.experiment.cross_validation(
        outcome_space=emotions,
        data_splits=splits,
        tokenizer=tokenizer_for_lstm,
        emb_model=trained_emb_model,
        enc_type='lstm', 
        enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.},
        regressor_kwargs={'hidden_size1': 100, 'hidden_size2': 100, 'dropout': 0.1},
        opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 20},    
        device='cpu', #'cpu'
        save_to=f"{results_folder}/lstm",
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.215 seconds
Dataloader done! Time taken: 0.026 seconds


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9267 | train acc: 0.1853 | val loss: 1.8708 | val acc: 0.2434
Epoch 02 | train loss: 1.7682 | train acc: 0.3081 | val loss: 1.6941 | val acc: 0.3324
Epoch 03 | train loss: 1.6114 | train acc: 0.3831 | val loss: 1.5329 | val acc: 0.4062
Epoch 04 | train loss: 1.5204 | train acc: 0.4238 | val loss: 1.4783 | val acc: 0.4328
Epoch 05 | train loss: 1.4484 | train acc: 0.4584 | val loss: 1.4261 | val acc: 0.4488
Epoch 06 | train loss: 1.3869 | train acc: 0.4802 | val loss: 1.3887 | val acc: 0.4814
Epoch 07 | train loss: 1.3415 | train acc: 0.5033 | val loss: 1.3847 | val acc: 0.4801
Epoch 08 | train loss: 1.3094 | train acc: 0.5163 | val loss: 1.3747 | val acc: 0.4860
Epoch 09 | train loss: 1.2784 | train acc: 0.5241 | val loss: 1.3141 | val acc: 0.5053
Epoch 10 | train loss: 1.2427 | train acc: 0.5394 | val loss: 1.2908 | val acc: 0.5186
Epoch 11 | train loss: 1.2166 | train acc: 0.5529 | val loss: 1.2970 | val acc: 0.5180
Epoch 12 | train loss: 1.1871 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9050 | train acc: 0.1976 | val loss: 1.7904 | val acc: 0.2768
Epoch 02 | train loss: 1.7110 | train acc: 0.3356 | val loss: 1.6308 | val acc: 0.3580
Epoch 03 | train loss: 1.5789 | train acc: 0.3951 | val loss: 1.5574 | val acc: 0.4052
Epoch 04 | train loss: 1.5087 | train acc: 0.4259 | val loss: 1.4904 | val acc: 0.4385
Epoch 05 | train loss: 1.4328 | train acc: 0.4578 | val loss: 1.4496 | val acc: 0.4558
Epoch 06 | train loss: 1.3877 | train acc: 0.4801 | val loss: 1.4510 | val acc: 0.4591
Epoch 07 | train loss: 1.3586 | train acc: 0.4946 | val loss: 1.4102 | val acc: 0.4850
Epoch 08 | train loss: 1.3066 | train acc: 0.5184 | val loss: 1.4084 | val acc: 0.4830
Epoch 09 | train loss: 1.2862 | train acc: 0.5242 | val loss: 1.3752 | val acc: 0.5096
Epoch 10 | train loss: 1.2438 | train acc: 0.5413 | val loss: 1.3377 | val acc: 0.5263
Epoch 11 | train loss: 1.2224 | train acc: 0.5545 | val loss: 1.3318 | val acc: 0.5163
Epoch 12 | train loss: 1.1981 | train acc: 

... (3 outputs left out)

Epoch 01 | train loss: 1.9043 | train acc: 0.1931 | val loss: 1.7823 | val acc: 0.2981
Epoch 02 | train loss: 1.7448 | train acc: 0.3057 | val loss: 1.7240 | val acc: 0.3067
Epoch 03 | train loss: 1.6316 | train acc: 0.3611 | val loss: 1.6094 | val acc: 0.3772
Epoch 04 | train loss: 1.5544 | train acc: 0.4050 | val loss: 1.5403 | val acc: 0.3979
Epoch 05 | train loss: 1.4879 | train acc: 0.4306 | val loss: 1.4883 | val acc: 0.4318
Epoch 06 | train loss: 1.4250 | train acc: 0.4572 | val loss: 1.4362 | val acc: 0.4631
Epoch 07 | train loss: 1.3845 | train acc: 0.4761 | val loss: 1.4063 | val acc: 0.4631
Epoch 08 | train loss: 1.3347 | train acc: 0.4966 | val loss: 1.3850 | val acc: 0.4890
Epoch 09 | train loss: 1.2994 | train acc: 0.5185 | val loss: 1.3308 | val acc: 0.5010
Epoch 10 | train loss: 1.2692 | train acc: 0.5304 | val loss: 1.3085 | val acc: 0.5216
Epoch 11 | train loss: 1.2367 | train acc: 0.5471 | val loss: 1.3004 | val acc: 0.5143
Epoch 12 | train loss: 1.2073 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9162 | train acc: 0.1866 | val loss: 1.8410 | val acc: 0.2695
Epoch 02 | train loss: 1.7423 | train acc: 0.3223 | val loss: 1.6287 | val acc: 0.3772
Epoch 03 | train loss: 1.6001 | train acc: 0.3908 | val loss: 1.5603 | val acc: 0.4005
Epoch 04 | train loss: 1.5118 | train acc: 0.4301 | val loss: 1.5184 | val acc: 0.4152
Epoch 05 | train loss: 1.4578 | train acc: 0.4527 | val loss: 1.4599 | val acc: 0.4491
Epoch 06 | train loss: 1.3963 | train acc: 0.4791 | val loss: 1.4617 | val acc: 0.4471
Epoch 07 | train loss: 1.3454 | train acc: 0.4971 | val loss: 1.4032 | val acc: 0.4717
Epoch 08 | train loss: 1.3115 | train acc: 0.5209 | val loss: 1.3794 | val acc: 0.4890
Epoch 09 | train loss: 1.2752 | train acc: 0.5370 | val loss: 1.3494 | val acc: 0.5017
Epoch 10 | train loss: 1.2382 | train acc: 0.5543 | val loss: 1.3517 | val acc: 0.4997
Epoch 11 | train loss: 1.2048 | train acc: 0.5608 | val loss: 1.3418 | val acc: 0.5090
Epoch 12 | train loss: 1.1834 | train acc: 

{'models': OrderedDict([(0, '../Data/Results/lstm/fold-0/model'),
              (1, '../Data/Results/lstm/fold-1/model'),
              (2, '../Data/Results/lstm/fold-2/model'),
              (3, '../Data/Results/lstm/fold-3/model'),
              (4, '../Data/Results/lstm/fold-4/model')]),
 'predictions': OrderedDict([(0,
               array([4, 1, 2, ..., 6, 3, 0], shape=(1504,))),
              (1, array([5, 1, 1, ..., 3, 4, 0], shape=(1503,))),
              (2, array([5, 0, 0, ..., 2, 0, 2], shape=(1503,))),
              (3, array([0, 1, 6, ..., 0, 2, 4], shape=(1503,))),
              (4, array([2, 2, 6, ..., 0, 0, 4], shape=(1503,)))]),
 'accuracies': OrderedDict([(0, 0.5585106382978723),
              (1, 0.5528942115768463),
              (2, 0.5382568196939455),
              (3, 0.5622089155023287),
              (4, 0.5548902195608783)])}

In [44]:
# dropout 0.1 -> 0.3
part2.experiment.cross_validation(
        outcome_space=emotions,
        data_splits=splits,
        tokenizer=tokenizer_for_lstm,
        emb_model=trained_emb_model,
        enc_type='lstm', 
        enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.},
        regressor_kwargs={'hidden_size1': 100, 'hidden_size2': 100, 'dropout': 0.3},
        opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 20},    
        device='cpu', #'cpu'
        save_to=f"{results_folder}/lstm",
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.327 seconds
Dataloader done! Time taken: 0.030 seconds


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9342 | train acc: 0.1753 | val loss: 1.8660 | val acc: 0.2374
Epoch 02 | train loss: 1.8166 | train acc: 0.2690 | val loss: 1.6803 | val acc: 0.3398
Epoch 03 | train loss: 1.6993 | train acc: 0.3300 | val loss: 1.6286 | val acc: 0.3710
Epoch 04 | train loss: 1.6158 | train acc: 0.3658 | val loss: 1.5247 | val acc: 0.4202
Epoch 05 | train loss: 1.5558 | train acc: 0.4009 | val loss: 1.5055 | val acc: 0.4315
Epoch 06 | train loss: 1.5084 | train acc: 0.4208 | val loss: 1.4456 | val acc: 0.4648
Epoch 07 | train loss: 1.4760 | train acc: 0.4405 | val loss: 1.4329 | val acc: 0.4594
Epoch 08 | train loss: 1.4358 | train acc: 0.4622 | val loss: 1.4013 | val acc: 0.4761
Epoch 09 | train loss: 1.4149 | train acc: 0.4709 | val loss: 1.3769 | val acc: 0.5027
Epoch 10 | train loss: 1.3798 | train acc: 0.4894 | val loss: 1.3829 | val acc: 0.4927
Epoch 11 | train loss: 1.3537 | train acc: 0.4990 | val loss: 1.3763 | val acc: 0.5086
Epoch 12 | train loss: 1.3250 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9349 | train acc: 0.1666 | val loss: 1.8903 | val acc: 0.2182
Epoch 02 | train loss: 1.8260 | train acc: 0.2584 | val loss: 1.7314 | val acc: 0.3194
Epoch 03 | train loss: 1.6834 | train acc: 0.3459 | val loss: 1.6074 | val acc: 0.3586
Epoch 04 | train loss: 1.5967 | train acc: 0.3857 | val loss: 1.5529 | val acc: 0.3832
Epoch 05 | train loss: 1.5287 | train acc: 0.4063 | val loss: 1.5022 | val acc: 0.4338
Epoch 06 | train loss: 1.4809 | train acc: 0.4445 | val loss: 1.4560 | val acc: 0.4371
Epoch 07 | train loss: 1.4233 | train acc: 0.4652 | val loss: 1.4299 | val acc: 0.4684
Epoch 08 | train loss: 1.3976 | train acc: 0.4763 | val loss: 1.4180 | val acc: 0.4637
Epoch 09 | train loss: 1.3726 | train acc: 0.4936 | val loss: 1.3839 | val acc: 0.4884
Epoch 10 | train loss: 1.3422 | train acc: 0.5051 | val loss: 1.3691 | val acc: 0.4990
Epoch 11 | train loss: 1.3135 | train acc: 0.5124 | val loss: 1.3651 | val acc: 0.4950
Epoch 12 | train loss: 1.3012 | train acc: 

... (3 outputs left out)

Epoch 01 | train loss: 1.9408 | train acc: 0.1683 | val loss: 1.9208 | val acc: 0.1683
Epoch 02 | train loss: 1.8159 | train acc: 0.2734 | val loss: 1.7721 | val acc: 0.2967
Epoch 03 | train loss: 1.6903 | train acc: 0.3363 | val loss: 1.6354 | val acc: 0.3526
Epoch 04 | train loss: 1.6144 | train acc: 0.3674 | val loss: 1.5815 | val acc: 0.3892
Epoch 05 | train loss: 1.5641 | train acc: 0.3860 | val loss: 1.5170 | val acc: 0.4225
Epoch 06 | train loss: 1.5101 | train acc: 0.4153 | val loss: 1.5147 | val acc: 0.4212
Epoch 07 | train loss: 1.4648 | train acc: 0.4484 | val loss: 1.4421 | val acc: 0.4524
Epoch 08 | train loss: 1.4302 | train acc: 0.4610 | val loss: 1.4109 | val acc: 0.4784
Epoch 09 | train loss: 1.3911 | train acc: 0.4816 | val loss: 1.3724 | val acc: 0.4930
Epoch 10 | train loss: 1.3686 | train acc: 0.4924 | val loss: 1.3838 | val acc: 0.5023
Epoch 11 | train loss: 1.3523 | train acc: 0.4901 | val loss: 1.3283 | val acc: 0.5057
Epoch 12 | train loss: 1.3102 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9425 | train acc: 0.1613 | val loss: 1.9158 | val acc: 0.1836
Epoch 02 | train loss: 1.8666 | train acc: 0.2401 | val loss: 1.7541 | val acc: 0.3187
Epoch 03 | train loss: 1.7586 | train acc: 0.3143 | val loss: 1.6582 | val acc: 0.3613
Epoch 04 | train loss: 1.6488 | train acc: 0.3630 | val loss: 1.5680 | val acc: 0.3952
Epoch 05 | train loss: 1.5621 | train acc: 0.3990 | val loss: 1.5136 | val acc: 0.4112
Epoch 06 | train loss: 1.5161 | train acc: 0.4246 | val loss: 1.5107 | val acc: 0.4378
Epoch 07 | train loss: 1.4580 | train acc: 0.4582 | val loss: 1.4252 | val acc: 0.4584
Epoch 08 | train loss: 1.4149 | train acc: 0.4735 | val loss: 1.4231 | val acc: 0.4644
Epoch 09 | train loss: 1.3815 | train acc: 0.4896 | val loss: 1.4217 | val acc: 0.4671
Epoch 10 | train loss: 1.3397 | train acc: 0.4989 | val loss: 1.3610 | val acc: 0.4877
Epoch 11 | train loss: 1.3184 | train acc: 0.5134 | val loss: 1.3355 | val acc: 0.5003
Epoch 12 | train loss: 1.2999 | train acc: 

{'models': OrderedDict([(0, '../Data/Results/lstm/fold-0/model'),
              (1, '../Data/Results/lstm/fold-1/model'),
              (2, '../Data/Results/lstm/fold-2/model'),
              (3, '../Data/Results/lstm/fold-3/model'),
              (4, '../Data/Results/lstm/fold-4/model')]),
 'predictions': OrderedDict([(0,
               array([4, 1, 5, ..., 3, 3, 0], shape=(1504,))),
              (1, array([5, 1, 1, ..., 3, 4, 0], shape=(1503,))),
              (2, array([5, 0, 0, ..., 2, 0, 2], shape=(1503,))),
              (3, array([4, 1, 6, ..., 0, 1, 4], shape=(1503,))),
              (4, array([6, 3, 2, ..., 0, 3, 4], shape=(1503,)))]),
 'accuracies': OrderedDict([(0, 0.5492021276595744),
              (1, 0.5349301397205589),
              (2, 0.5402528276779773),
              (3, 0.5442448436460412),
              (4, 0.5575515635395875)])}

In [45]:
# weight_decay 1e-4 -> 1e-3
part2.experiment.cross_validation(
        outcome_space=emotions,
        data_splits=splits,
        tokenizer=tokenizer_for_lstm,
        emb_model=trained_emb_model,
        enc_type='lstm', 
        enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.},
        regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
        opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-3, 'batch_size': 64, 'num_epochs': 20},    
        device='cpu', #'cpu'
        save_to=f"{results_folder}/lstm",
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.304 seconds
Dataloader done! Time taken: 0.025 seconds


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9396 | train acc: 0.1700 | val loss: 1.9022 | val acc: 0.2068
Epoch 02 | train loss: 1.8061 | train acc: 0.2874 | val loss: 1.6561 | val acc: 0.3584
Epoch 03 | train loss: 1.6331 | train acc: 0.3723 | val loss: 1.5517 | val acc: 0.4016
Epoch 04 | train loss: 1.5342 | train acc: 0.4170 | val loss: 1.5046 | val acc: 0.4408
Epoch 05 | train loss: 1.4687 | train acc: 0.4484 | val loss: 1.4519 | val acc: 0.4661
Epoch 06 | train loss: 1.4097 | train acc: 0.4802 | val loss: 1.4088 | val acc: 0.4940
Epoch 07 | train loss: 1.3550 | train acc: 0.4993 | val loss: 1.3793 | val acc: 0.5013
Epoch 08 | train loss: 1.3243 | train acc: 0.5185 | val loss: 1.3385 | val acc: 0.5133
Epoch 09 | train loss: 1.2744 | train acc: 0.5404 | val loss: 1.3369 | val acc: 0.5186
Epoch 10 | train loss: 1.2451 | train acc: 0.5579 | val loss: 1.3018 | val acc: 0.5359
Epoch 11 | train loss: 1.2110 | train acc: 0.5667 | val loss: 1.3038 | val acc: 0.5479
Epoch 12 | train loss: 1.1898 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9416 | train acc: 0.1635 | val loss: 1.9158 | val acc: 0.2255
Epoch 02 | train loss: 1.8040 | train acc: 0.2837 | val loss: 1.6933 | val acc: 0.3280
Epoch 03 | train loss: 1.6600 | train acc: 0.3554 | val loss: 1.6082 | val acc: 0.3619
Epoch 04 | train loss: 1.5665 | train acc: 0.3985 | val loss: 1.5349 | val acc: 0.4125
Epoch 05 | train loss: 1.5000 | train acc: 0.4186 | val loss: 1.5024 | val acc: 0.4178
Epoch 06 | train loss: 1.4454 | train acc: 0.4542 | val loss: 1.4448 | val acc: 0.4518
Epoch 07 | train loss: 1.3976 | train acc: 0.4731 | val loss: 1.4271 | val acc: 0.4571
Epoch 08 | train loss: 1.3525 | train acc: 0.4981 | val loss: 1.3969 | val acc: 0.4817
Epoch 09 | train loss: 1.3250 | train acc: 0.5077 | val loss: 1.4141 | val acc: 0.4697
Epoch 10 | train loss: 1.2959 | train acc: 0.5255 | val loss: 1.3807 | val acc: 0.4990
Epoch 11 | train loss: 1.2647 | train acc: 0.5360 | val loss: 1.3603 | val acc: 0.5090
Epoch 12 | train loss: 1.2338 | train acc: 

... (3 outputs left out)

Epoch 01 | train loss: 1.9330 | train acc: 0.1713 | val loss: 1.8733 | val acc: 0.2255
Epoch 02 | train loss: 1.7998 | train acc: 0.2747 | val loss: 1.7588 | val acc: 0.2954
Epoch 03 | train loss: 1.6800 | train acc: 0.3371 | val loss: 1.6389 | val acc: 0.3540
Epoch 04 | train loss: 1.5930 | train acc: 0.3835 | val loss: 1.5961 | val acc: 0.3919
Epoch 05 | train loss: 1.5164 | train acc: 0.4055 | val loss: 1.5159 | val acc: 0.4258
Epoch 06 | train loss: 1.4701 | train acc: 0.4364 | val loss: 1.4657 | val acc: 0.4338
Epoch 07 | train loss: 1.4067 | train acc: 0.4534 | val loss: 1.4078 | val acc: 0.4824
Epoch 08 | train loss: 1.3722 | train acc: 0.4790 | val loss: 1.4118 | val acc: 0.4770
Epoch 09 | train loss: 1.3276 | train acc: 0.5084 | val loss: 1.3677 | val acc: 0.4970
Epoch 10 | train loss: 1.2941 | train acc: 0.5157 | val loss: 1.3464 | val acc: 0.5057
Epoch 11 | train loss: 1.2618 | train acc: 0.5307 | val loss: 1.3283 | val acc: 0.5203
Epoch 12 | train loss: 1.2430 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9313 | train acc: 0.1853 | val loss: 1.8687 | val acc: 0.2608
Epoch 02 | train loss: 1.8026 | train acc: 0.2925 | val loss: 1.7196 | val acc: 0.3393
Epoch 03 | train loss: 1.6396 | train acc: 0.3764 | val loss: 1.5530 | val acc: 0.4172
Epoch 04 | train loss: 1.5304 | train acc: 0.4257 | val loss: 1.4919 | val acc: 0.4464
Epoch 05 | train loss: 1.4600 | train acc: 0.4499 | val loss: 1.4644 | val acc: 0.4744
Epoch 06 | train loss: 1.4029 | train acc: 0.4778 | val loss: 1.4299 | val acc: 0.4644
Epoch 07 | train loss: 1.3430 | train acc: 0.5121 | val loss: 1.3705 | val acc: 0.5010
Epoch 08 | train loss: 1.3076 | train acc: 0.5230 | val loss: 1.3440 | val acc: 0.5130
Epoch 09 | train loss: 1.2683 | train acc: 0.5430 | val loss: 1.3362 | val acc: 0.5190
Epoch 10 | train loss: 1.2523 | train acc: 0.5515 | val loss: 1.3122 | val acc: 0.5316
Epoch 11 | train loss: 1.2199 | train acc: 0.5643 | val loss: 1.3009 | val acc: 0.5383
Epoch 12 | train loss: 1.1863 | train acc: 

{'models': OrderedDict([(0, '../Data/Results/lstm/fold-0/model'),
              (1, '../Data/Results/lstm/fold-1/model'),
              (2, '../Data/Results/lstm/fold-2/model'),
              (3, '../Data/Results/lstm/fold-3/model'),
              (4, '../Data/Results/lstm/fold-4/model')]),
 'predictions': OrderedDict([(0,
               array([4, 1, 6, ..., 3, 3, 0], shape=(1504,))),
              (1, array([5, 1, 1, ..., 3, 2, 0], shape=(1503,))),
              (2, array([5, 0, 2, ..., 2, 0, 2], shape=(1503,))),
              (3, array([4, 4, 3, ..., 3, 1, 4], shape=(1503,))),
              (4, array([6, 2, 6, ..., 0, 3, 4], shape=(1503,)))]),
 'accuracies': OrderedDict([(0, 0.5757978723404256),
              (1, 0.5382568196939455),
              (2, 0.5382568196939455),
              (3, 0.552228875582169),
              (4, 0.5602129075182968)])}

In [46]:
# learning rate 1e-4 -> 5e-5
part2.experiment.cross_validation(
        outcome_space=emotions,
        data_splits=splits,
        tokenizer=tokenizer_for_lstm,
        emb_model=trained_emb_model,
        enc_type='lstm', 
        enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.},
        regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
        opt_kwargs={'learning_rate': 5e-5, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 20},    
        device='cpu', #'cpu'
        save_to=f"{results_folder}/lstm",
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.200 seconds
Dataloader done! Time taken: 0.024 seconds


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9475 | train acc: 0.1482 | val loss: 1.9480 | val acc: 0.1596
Epoch 02 | train loss: 1.9457 | train acc: 0.1542 | val loss: 1.9460 | val acc: 0.1908
Epoch 03 | train loss: 1.9438 | train acc: 0.1670 | val loss: 1.9437 | val acc: 0.2001
Epoch 04 | train loss: 1.9413 | train acc: 0.1766 | val loss: 1.9406 | val acc: 0.2021
Epoch 05 | train loss: 1.9395 | train acc: 0.1758 | val loss: 1.9362 | val acc: 0.2094
Epoch 06 | train loss: 1.9353 | train acc: 0.1944 | val loss: 1.9301 | val acc: 0.2174
Epoch 07 | train loss: 1.9286 | train acc: 0.2023 | val loss: 1.9209 | val acc: 0.2314
Epoch 08 | train loss: 1.9200 | train acc: 0.2139 | val loss: 1.9078 | val acc: 0.2487
Epoch 09 | train loss: 1.9060 | train acc: 0.2279 | val loss: 1.8908 | val acc: 0.2600
Epoch 10 | train loss: 1.8904 | train acc: 0.2372 | val loss: 1.8716 | val acc: 0.2640
Epoch 11 | train loss: 1.8757 | train acc: 0.2420 | val loss: 1.8562 | val acc: 0.2719
Epoch 12 | train loss: 1.8618 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9478 | train acc: 0.1552 | val loss: 1.9488 | val acc: 0.1430
Epoch 02 | train loss: 1.9459 | train acc: 0.1590 | val loss: 1.9469 | val acc: 0.1437
Epoch 03 | train loss: 1.9438 | train acc: 0.1608 | val loss: 1.9451 | val acc: 0.1524
Epoch 04 | train loss: 1.9425 | train acc: 0.1668 | val loss: 1.9433 | val acc: 0.1583
Epoch 05 | train loss: 1.9405 | train acc: 0.1710 | val loss: 1.9413 | val acc: 0.1623
Epoch 06 | train loss: 1.9379 | train acc: 0.1758 | val loss: 1.9391 | val acc: 0.1697
Epoch 07 | train loss: 1.9347 | train acc: 0.1856 | val loss: 1.9357 | val acc: 0.1816
Epoch 08 | train loss: 1.9311 | train acc: 0.1976 | val loss: 1.9311 | val acc: 0.2016
Epoch 09 | train loss: 1.9246 | train acc: 0.2029 | val loss: 1.9239 | val acc: 0.2142
Epoch 10 | train loss: 1.9136 | train acc: 0.2219 | val loss: 1.9100 | val acc: 0.2322
Epoch 11 | train loss: 1.8946 | train acc: 0.2345 | val loss: 1.8884 | val acc: 0.2468
Epoch 12 | train loss: 1.8684 | train acc: 

... (3 outputs left out)

Epoch 01 | train loss: 1.9522 | train acc: 0.1394 | val loss: 1.9489 | val acc: 0.1364
Epoch 02 | train loss: 1.9491 | train acc: 0.1385 | val loss: 1.9468 | val acc: 0.1364
Epoch 03 | train loss: 1.9468 | train acc: 0.1440 | val loss: 1.9453 | val acc: 0.1464
Epoch 04 | train loss: 1.9456 | train acc: 0.1449 | val loss: 1.9443 | val acc: 0.1577
Epoch 05 | train loss: 1.9449 | train acc: 0.1488 | val loss: 1.9431 | val acc: 0.1570
Epoch 06 | train loss: 1.9429 | train acc: 0.1636 | val loss: 1.9416 | val acc: 0.1690
Epoch 07 | train loss: 1.9415 | train acc: 0.1675 | val loss: 1.9395 | val acc: 0.1756
Epoch 08 | train loss: 1.9392 | train acc: 0.1774 | val loss: 1.9366 | val acc: 0.1836
Epoch 09 | train loss: 1.9363 | train acc: 0.1794 | val loss: 1.9321 | val acc: 0.1863
Epoch 10 | train loss: 1.9308 | train acc: 0.1903 | val loss: 1.9256 | val acc: 0.1976
Epoch 11 | train loss: 1.9250 | train acc: 0.1992 | val loss: 1.9173 | val acc: 0.2089
Epoch 12 | train loss: 1.9146 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9515 | train acc: 0.1462 | val loss: 1.9519 | val acc: 0.1264
Epoch 02 | train loss: 1.9477 | train acc: 0.1463 | val loss: 1.9488 | val acc: 0.1264
Epoch 03 | train loss: 1.9465 | train acc: 0.1482 | val loss: 1.9467 | val acc: 0.1271
Epoch 04 | train loss: 1.9442 | train acc: 0.1465 | val loss: 1.9448 | val acc: 0.1311
Epoch 05 | train loss: 1.9427 | train acc: 0.1538 | val loss: 1.9426 | val acc: 0.1497
Epoch 06 | train loss: 1.9408 | train acc: 0.1625 | val loss: 1.9400 | val acc: 0.1657
Epoch 07 | train loss: 1.9380 | train acc: 0.1748 | val loss: 1.9362 | val acc: 0.2016
Epoch 08 | train loss: 1.9325 | train acc: 0.1942 | val loss: 1.9301 | val acc: 0.2262
Epoch 09 | train loss: 1.9262 | train acc: 0.2105 | val loss: 1.9198 | val acc: 0.2508
Epoch 10 | train loss: 1.9145 | train acc: 0.2240 | val loss: 1.9038 | val acc: 0.2535
Epoch 11 | train loss: 1.8953 | train acc: 0.2421 | val loss: 1.8845 | val acc: 0.2681
Epoch 12 | train loss: 1.8782 | train acc: 

{'models': OrderedDict([(0, '../Data/Results/lstm/fold-0/model'),
              (1, '../Data/Results/lstm/fold-1/model'),
              (2, '../Data/Results/lstm/fold-2/model'),
              (3, '../Data/Results/lstm/fold-3/model'),
              (4, '../Data/Results/lstm/fold-4/model')]),
 'predictions': OrderedDict([(0,
               array([4, 1, 6, ..., 6, 6, 0], shape=(1504,))),
              (1, array([1, 3, 1, ..., 3, 4, 0], shape=(1503,))),
              (2, array([6, 0, 1, ..., 3, 3, 6], shape=(1503,))),
              (3, array([6, 6, 2, ..., 1, 0, 2], shape=(1503,))),
              (4, array([6, 4, 6, ..., 0, 0, 1], shape=(1503,)))]),
 'accuracies': OrderedDict([(0, 0.3098404255319149),
              (1, 0.27811044577511645),
              (2, 0.26413838988689287),
              (3, 0.2860944777112442),
              (4, 0.2940785096473719)])}